# Import Libraries

In [ ]:
!pip install git+https://github.com/jacobgil/pytorch-grad-cam.git


In [ ]:
!pip install lime

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.transforms import ToTensor
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import zipfile
import os
from tqdm import tqdm  # Import tqdm for progress bars
from PIL import Image
import seaborn as sns
import pandas as pd
import random

import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode
from PIL import Image, ImageFilter
from collections import defaultdict
from torch.utils.data import Dataset 
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix
from lime.lime_image import LimeImageExplainer
from skimage.segmentation import mark_boundaries

from tqdm.notebook import tqdm
import torch.amp as amp

# Load Dataset

In [ ]:
data_dir = "/kaggle/input/mulberry-leaf-dataset/Mulberry Leaf Dataset/Original"
classes = os.listdir(data_dir)

print(f"Total classes: {len(classes)}")

for cls in classes:
    print(f"Class: {cls}, Number of Image: {len(os.listdir(os.path.join(data_dir, cls)))}")

# Data Preprocessing

## Display unique image sizes across all classes

In [ ]:
from PIL import Image

image_sizes = []
for cls in classes:
    for img_file in os.listdir(os.path.join(data_dir, cls)):
        img = Image.open(os.path.join(data_dir, cls, img_file))
        image_sizes.append(img.size)

print(set(image_sizes))

## Plotting number of images per class using a bar chart

In [ ]:
class_counts = [len(os.listdir(os.path.join(data_dir, cls))) for cls in classes]

# DataFrame
df = pd.DataFrame({'Classes': classes, 'Class_Count': class_counts})

sns.barplot(data=df, x='Classes', y='Class_Count')
plt.xticks(rotation=45)
plt.title("Class Distribution")
plt.show()

## Showing random image samples from each class

In [ ]:
fig, axes = plt.subplots(len(classes), 5, figsize=(15, len(classes)*3))

for i, cls in enumerate(classes):
    class_dir = os.path.join(data_dir, cls)
    for j in range(5):
        img_path = os.path.join(class_dir, random.choice(os.listdir(class_dir)))
        img = Image.open(img_path)
        axes[i,j].imshow(img)
        axes[i,j].axis("off")
        if j==0:
            axes[i,j].set_title(cls)
plt.show()

## Detect Corrupted Images

In [ ]:
corrupted_images = []
for cls in classes:
    for img_file in os.listdir(os.path.join(data_dir, cls)):
        try:
            img = Image.open(os.path.join(data_dir, cls, img_file))
            img.verify()
        except (IOError, SyntaxError):
            corrupted_images.append(img_file)
print("Corrupted Images: ",corrupted_images)

# Data Augmentation & Dataset Splitting

In [ ]:
def unsharp_mask(image):
    return image.filter(ImageFilter.UnsharpMask(radius=2, percent=150, threshold=3))

transform_train = transforms.Compose([
    transforms.RandomResizedCrop(size=224, scale=(0.8, 1.0), interpolation=InterpolationMode.BICUBIC),
    transforms.Lambda(unsharp_mask),
    transforms.RandomHorizontalFlip(),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.5),(0.5))
])

transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5),(0.5))
])

dataset = datasets.ImageFolder(root=data_dir, transform=transform_train)

# Get class counts
class_counts = [0] * len(dataset.classes)
for _, label in dataset:
    class_counts[label] += 1

target_count = 1500 # Target number of images per class

oversampled_samples = []
class_data = defaultdict(list)
for img_path, label in dataset.imgs:
    class_data[label].append((img_path, label))

for label, samples in class_data.items():
    count = len(samples)
    oversampled_samples.extend(samples)
    if count < target_count:
        num_to_add = target_count - count
        oversampled_samples.extend(random.choices(samples, k=num_to_add))

# Create a custom dataset from the oversampled samples
class BalancedDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# Create the balanced dataset
balanced_dataset = BalancedDataset(oversampled_samples, transform=transform_train)


# Split the balanced dataset into train, validation, and test sets
train_size = int(0.7 * len(balanced_dataset))
val_size = int(0.2 * len(balanced_dataset))
test_size = len(balanced_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(balanced_dataset, [train_size, val_size, test_size])

# Apply the test transform to the validation and test sets
val_dataset.dataset.transform = transform_test
test_dataset.dataset.transform = transform_test


# Create DataLoaders with the specified batch size
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Balanced dataset created and split into train, validation, and test sets.")

In [ ]:
# Get the class counts from the balanced dataset
balanced_class_counts = [0] * len(dataset.classes)
for _, label in balanced_dataset:
    balanced_class_counts[label] += 1

# Create a DataFrame for plotting
df_balanced = pd.DataFrame({'Classes': dataset.classes, 'Class_Count': balanced_class_counts})

# Plotting the class distribution
plt.figure(figsize=(10, 6))
sns.barplot(data=df_balanced, x='Classes', y='Class_Count')
plt.xticks(rotation=45)
plt.title("Balanced Class Distribution (Oversampled)")
plt.xlabel("Classes")
plt.ylabel("Number of Images")
plt.tight_layout()
plt.show()

## Visualize One Augmented Image per Class

In [ ]:
class_names = dataset.classes
num_classes = len(class_names)

fig, axs = plt.subplots(2, num_classes // 2, figsize=(15, 5))
axs = axs.flatten()

displayed_classes = {class_name: False for class_name in class_names}

# Loop through data to get one image per class
for images, labels in train_loader:
    for img, label in zip(images, labels):
        class_name = class_names[label]
        if not displayed_classes[class_name]:
            img = img.permute(1, 2, 0).numpy()
            img = (img * 0.5) + 0.5
            axs[label].imshow(np.clip(img, 0, 1))
            axs[label].set_title(class_name)
            axs[label].axis('off')
            displayed_classes[class_name] = True

        if all(displayed_classes.values()):
            break
    if all(displayed_classes.values()):
        break

plt.tight_layout()
plt.show()

# Transfer Learning Models

In [ ]:
def get_transfer_model(model_name, num_classes):
    if model_name == "regnet_y_128gf":
        model = models.regnet_y_128gf(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "vit_l_16":
        model = models.vit_l_16(pretrained=True)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    elif model_name == "wide_resnet101_2":
        model = models.wide_resnet101_2(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "vgg19":
        model = models.vgg19(pretrained=True)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif model_name == "resnet152":
        model = models.resnet152(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "resnext101_32x8d":
        model = models.resnext101_32x8d(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "swin_b":
        model = models.swin_b(pretrained=True)
        model.head = nn.Linear(model.head.in_features, num_classes)
    elif model_name == "regnet_y_32gf":
        model = models.regnet_y_32gf(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "vit_l_32":
        model = models.vit_l_32(pretrained=True)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    elif model_name == "regnet_y_16gf":
        model = models.regnet_y_16gf(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "resnet50":
        model = models.resnet50(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        raise ValueError(f"Model '{model_name}' not supported. Choose from regnet_y_128gf, vit_l_16, wide_resnet101_2, vgg19, resnet152, resnext101_32x8d, swin_b, regnet_y_32gf, vit_l_32, regnet_y_16gf.")
    return model

## Early Stopping

In [ ]:
class EarlyStopping:

    def __init__(self, patience=5):

        self.patience = patience
        self.counter = 0
        self.best_loss = np.inf

    def check_early_stop(self, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False


# Transfer Learning Example using resnet50

In [ ]:
# Set number of epochs and initialize variables
num_epochs = 50  # Define the number of training epochs

model = get_transfer_model('resnet50', num_classes).to('cuda')
criterion = nn.CrossEntropyLoss()  # Loss function

optimizer = optim.Adam(model.parameters(), lr=0.001)  # Optimizer with learning rate

early_stopping = EarlyStopping(patience=5)  # Initialize early stopping with patience

train_losses, val_losses = [], []  # Lists to store training and validation losses per epoch

# Initialize GradScaler for AMP
scaler = GradScaler()
# Loop over epochs
for epoch in range(num_epochs):

    print(f"Epoch {epoch+1}/{num_epochs}")
    # Training phase
    model.train()  # Set model to training mode
    train_loss = 0  # Initialize cumulative training loss for the epoch
    # Loop over training data with tqdm progress bar
    for images, labels in tqdm(train_loader, desc="Training", leave=False):
        images, labels = images.to('cuda'), labels.to('cuda')  # Move data to GPU

        optimizer.zero_grad()  # Clear previous gradients

        with autocast():  # Use AMP for mixed-precision calculations
            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss

        # Scale loss to avoid underflow for float16
        scaler.scale(loss).backward()  # Backward pass with scaled loss
        scaler.step(optimizer)  # Optimizer step
        scaler.update()  # Update the scaler for next iteration
        train_loss += loss.item()  # Accumulate the training loss

    # Validation phase
    model.eval()  # Set model to evaluation mode
    val_loss = 0  # Initialize cumulative validation loss for the epoch
    with torch.no_grad():  # Disable gradient calculation for validation
        # Loop over validation data with tqdm progress bar
        for images, labels in tqdm(val_loader, desc="Validation", leave=False):
            images, labels = images.to('cuda'), labels.to('cuda')  # Move data to GPU
            with autocast():  # Use AMP for mixed-precision calculations
                outputs = model(images)  # Forward pass
                loss = criterion(outputs, labels)  # Compute loss
            val_loss += loss.item()  # Accumulate the validation loss

    # Calculate average losses and append to lists
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    print(f"Train Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}")

    # Early stopping check
    if early_stopping.check_early_stop(avg_val_loss):
        print("Early stopping triggered.")
        break


## Save model

In [ ]:
# Save the model
torch.save(model.state_dict(), 'transfer_learning_resnet50.pth')
print("Model saved as 'transfer_learning_resnet50.pth'")

## Validation Loss Curves

In [ ]:
# Plotting loss curves
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title("Training and Validation Loss")
plt.show()

## Model Performance

In [ ]:
# Move model to evaluation mode
model.eval()

# Initialize lists to store predictions and true labels
all_preds = []
all_labels = []

# Run inference
with torch.no_grad():  # Disable gradient calculation for faster inference
    for images, labels in test_loader:
        # Move images and labels to the same device as the model (GPU)
        images, labels = images.to('cuda'), labels.to('cuda')

        # Forward pass
        outputs = model(images)

        # Get predictions (choose the class with the highest logit score)
        _, predicted = torch.max(outputs, 1)

        # Append predictions and true labels to lists
        all_preds.extend(predicted.cpu().numpy())  # Move to CPU and convert to numpy
        all_labels.extend(labels.cpu().numpy())    # Move to CPU and convert to numpy

# Calculate evaluation metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='weighted')
recall = recall_score(all_labels, all_preds, average='weighted')
f1 = f1_score(all_labels, all_preds, average='weighted')

# Print the evaluation metrics
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

## Confusion Matrix

In [ ]:
# Calculate the confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# Display the confusion matrix using a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# XAI

In [ ]:
# Select a random image path and label from the test_dataset
random_index = random.randint(0, len(test_dataset) - 1)
image_path, true_label = test_dataset.dataset.samples[test_dataset.indices[random_index]]

# Open and preprocess the image
image = Image.open(image_path).convert("RGB")
input_tensor = transform_test(image)
input_tensor = input_tensor.unsqueeze(0)  # Add batch dimension
input_tensor = input_tensor.to('cuda')  # Move to GPU

print(f"Selected image: {image_path}")
print(f"True label: {true_label}")

## Grad-cam

In [ ]:
# Define the target layer for Grad-CAM
target_layer = model.layer4[-1]  # Last Bottleneck block of layer4

# Instantiate the GradCAM object
grad_cam = GradCAM(model=model, target_layers=[target_layer])

# Compute the heatmap
grayscale_cam = grad_cam(input_tensor=input_tensor, targets=None)

# In this example, we will reshape the heatmap to 2D
grayscale_cam = grayscale_cam[0, :]

print("Grad-CAM heatmap computed.")

## Grad-cam ++

In [ ]:
# Define the target layer for Grad-CAM++
target_layer = model.layer4[-1]  # Last Bottleneck block of layer4

# Instantiate the GradCAMPlusPlus object
grad_cam_plus_plus = GradCAMPlusPlus(model=model, target_layers=[target_layer])

# Compute the heatmap
grayscale_cam_plus_plus = grad_cam_plus_plus(input_tensor=input_tensor, targets=None)

# Extract the 2D heatmap
grayscale_cam_plus_plus = grayscale_cam_plus_plus[0, :]

print("Grad-CAM++ heatmap computed.")

## Eigen-cam

In [ ]:
# Define the target layer for Eigen-CAM
target_layer = model.layer4[-1]  # Last Bottleneck block of layer4

# Instantiate the EigenCAM object
eigen_cam = EigenCAM(model=model, target_layers=[target_layer])

# Compute the heatmap
eigen_grayscale_cam = eigen_cam(input_tensor=input_tensor, targets=None)

# Extract the 2D heatmap
eigen_grayscale_cam = eigen_grayscale_cam[0, :]

print("Eigen-CAM heatmap computed.")

## Ablation-cam

In [ ]:
from pytorch_grad_cam import AblationCAM

# Define the target layer for Ablation-CAM
target_layer = model.layer4[-1]  # Last Bottleneck block of layer4

# Instantiate the AblationCAM object
ablation_cam = AblationCAM(model=model, target_layers=[target_layer])

# Compute the heatmap
ablation_grayscale_cam = ablation_cam(input_tensor=input_tensor, targets=None)

# Extract the 2D heatmap
ablation_grayscale_cam = ablation_grayscale_cam[0, :]

print("Ablation-CAM heatmap computed.")

## Visualize results


In [ ]:
# Convert the original image to a NumPy array in the range [0, 1]
original_image_np = np.array(image).astype(np.float32) / 255.0

# Overlay heatmaps on the original image
grad_cam_overlay = show_cam_on_image(original_image_np, grayscale_cam, use_rgb=True)
grad_cam_plus_plus_overlay = show_cam_on_image(original_image_np, grayscale_cam_plus_plus, use_rgb=True)
eigen_cam_overlay = show_cam_on_image(original_image_np, eigen_grayscale_cam, use_rgb=True)
ablation_cam_overlay = show_cam_on_image(original_image_np, ablation_grayscale_cam, use_rgb=True)

# Create subplots to display the images
fig, axes = plt.subplots(1, 5, figsize=(25, 5))

# Plot the original image
axes[0].imshow(original_image_np)
axes[0].set_title("Original Image")
axes[0].axis('off')

# Plot the Grad-CAM overlay
axes[1].imshow(grad_cam_overlay)
axes[1].set_title("Grad-CAM")
axes[1].axis('off')

# Plot the Grad-CAM++ overlay
axes[2].imshow(grad_cam_plus_plus_overlay)
axes[2].set_title("Grad-CAM++")
axes[2].axis('off')

# Plot the Eigen-CAM overlay
axes[3].imshow(eigen_cam_overlay)
axes[3].set_title("Eigen-CAM")
axes[3].axis('off')

# Plot the Ablation-CAM overlay (last)
axes[4].imshow(ablation_cam_overlay)
axes[4].set_title("Ablation-CAM")
axes[4].axis('off')

# Adjust layout and display the plot
plt.tight_layout()
plt.show()

# LIME

In [ ]:
from lime.lime_image import LimeImageExplainer

# Instantiate the LimeImageExplainer
explainer = LimeImageExplainer()

print("LIME image explainer set up.")

In [ ]:
# Convert the PIL Image to a NumPy array
image_np = np.array(image)

# Ensure the NumPy array is in the format (height, width, channels) and pixel values are in [0, 255]
# PIL Image is already in (height, width, channels) and values are in [0, 255] for RGB images.
# We can verify the shape and data type.
print(f"Image shape: {image_np.shape}")
print(f"Image data type: {image_np.dtype}")

In [ ]:
def batch_predict(images):
    # Convert the NumPy array of images to a PyTorch tensor
    images = torch.from_numpy(images).permute(0, 3, 1, 2).float().to('cuda')

    # Set the model to evaluation mode and get predictions
    model.eval()
    with torch.no_grad():
        outputs = model(images)

    # Apply softmax to get probabilities and convert to NumPy
    probabilities = torch.softmax(outputs, dim=1).cpu().numpy()
    return probabilities

# Generate LIME explanation
explanation = explainer.explain_instance(
    image_np,  # The NumPy array of the image
    batch_predict, # The batch prediction function
    top_labels=5, # Explain the top 5 predicted classes
    hide_color=0, # Hide color for perturbed images
    num_samples=1000 # Number of perturbed samples to generate
)

print("LIME explanation generated.")

In [ ]:
# Select the top prediction from the explanation
top_pred_label = explanation.top_labels[0]

# Get the image and mask for the explanation of the top class
temp, mask = explanation.get_image_and_mask(
    top_pred_label, positive_only=True, num_features=5, hide_rest=False  # Set hide_rest to False
)

# Overlay the explanation mask on the original image
# Use the original image_np (scaled to [0, 1]) as the base for mark_boundaries
lime_overlay = mark_boundaries(image_np.astype(np.float64) / 255.0, mask)


# Create a figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot the original image
axes[0].imshow(image_np)
axes[0].set_title("Original Image")
axes[0].axis('off')

# Plot the LIME explanation overlay
axes[1].imshow(lime_overlay)
axes[1].set_title("LIME Explanation Overlay") # Updated title for clarity
axes[1].axis('off')

# Adjust layout and display the plot
plt.tight_layout()
plt.show()